# Week 03 · SQL、事务、索引与 SQLAlchemy

关系数据库把数据组织为表；主键唯一标识一行，外键表达引用关系。一个用户有多个任务：users.id ← tasks.user_id。NOT NULL、UNIQUE、CHECK 和 FOREIGN KEY 是最后一道数据一致性约束，不能只靠页面校验。SQLite 默认可能不开外键，所以每条新连接显式设置 PRAGMA。

事务把一组修改作为一个整体：成功提交，失败回滚。例如转移任务与写审计日志必须同时成功。索引是额外的数据结构，能减少读取扫描，但需要空间并增加写入成本。用 EXPLAIN QUERY PLAN 检查实际执行计划；不要因为建了索引就假定每次查询都更快。分页必须有稳定排序，否则相同请求可能翻出重复/遗漏结果。

ORM 把 Python 对象映射到表，不能替你理解 SQL。Session 是工作单元，离开事务后要避免继续依赖懒加载。生产演进使用 Alembic 记录迁移；create_all 只创建缺失表，不是完整迁移工具。SQLite 与 PostgreSQL 的并发、类型和部署模型不同，上线之前要用真实目标数据库集成测试。

## 学习方式 / How to study
先预测代码结果，再逐行运行。改变一个输入、解释变化，最后不看参考实现重写关键函数。阅读不是掌握的证据；能独立实现、测试、解释失败才是。

In [ ]:
from sqlalchemy import create_engine, event, ForeignKey, String, select, text
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, Session
from sqlalchemy.exc import IntegrityError

class Base(DeclarativeBase): pass
class User(Base):
    __tablename__ = "users"
    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(100), unique=True)
class Task(Base):
    __tablename__ = "tasks"
    id: Mapped[int] = mapped_column(primary_key=True)
    user_id: Mapped[int] = mapped_column(ForeignKey("users.id"), index=True)
    title: Mapped[str] = mapped_column(String(120))

engine = create_engine("sqlite:///week03-demo.db")
@event.listens_for(engine, "connect")
def constraints(connection, _):
    connection.execute("PRAGMA foreign_keys=ON")
Base.metadata.create_all(engine)
# 只清理当前示例表的数据，使实验重复运行得到相同结果。
with engine.begin() as connection:
    connection.execute(text("DELETE FROM tasks"))
    connection.execute(text("DELETE FROM users"))
with Session(engine) as session, session.begin():
    user = User(name="Carter")
    session.add(user)
    session.flush()  # 获得主键，但此时事务尚未提交。
    session.add_all([Task(user_id=user.id, title=f"Task {i}") for i in range(8)])
with Session(engine) as session:
    query = select(Task.title, User.name).join(User).order_by(Task.id).offset(2).limit(3)
    print("JOIN + 稳定分页：", session.execute(query).all())
try:
    with Session(engine) as session, session.begin():
        session.add(User(name="RolledBack"))
        session.add(Task(user_id=999999, title="Invalid foreign key"))
except IntegrityError:
    print("外键错误导致整笔事务回滚")
with Session(engine) as session:
    assert session.scalar(select(User).where(User.name == "RolledBack")) is None
with engine.connect() as connection:
    print(connection.execute(text("EXPLAIN QUERY PLAN SELECT * FROM tasks WHERE user_id=1")).all())
engine.dispose()
# 新建连接模拟服务重启后的重新读取，而不是依赖内存中的 ORM 对象。
reopened = create_engine("sqlite:///week03-demo.db")
with reopened.connect() as connection:
    assert connection.execute(text("SELECT count(*) FROM tasks")).scalar() == 8
print("持久化、JOIN、分页、回滚均通过")

## 练习 / Exercises
写 GROUP BY 统计每个人的任务数；考虑没有任务的用户为什么需要 LEFT JOIN。

先在下面独立完成，再展开参考实现。

In [ ]:
# 在这里写你的实现；运行后检查边界。


## 参考实现与验收 / Reference and checks
参考实现是一个可行方案，不是唯一答案。不要在未完成练习前直接复制。

In [ ]:
with reopened.connect() as connection:
    rows = connection.execute(text("SELECT users.name, count(tasks.id) FROM users LEFT JOIN tasks ON tasks.user_id=users.id GROUP BY users.id ORDER BY users.id")).all()
    print(rows)
    assert rows[0][1] == 8
reopened.dispose()